### Imports des bibliothèques

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
import scipy.io
import os
import subprocess
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset
from torch.distributions import Dirichlet
import random
import collections
import sklearn.metrics
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
import pandas as pd

### Configuration de l'environnement

In [2]:
# Dossier WLDA DEDIE
WLDA_ROOT = r"C:\Users\Home\Documents\M2\Projet_PPD\output\WLDA"
DATA_ROOT = r"C:\Users\Home\Documents\M2\Projet_PPD\data"

SEED = 42
TOPN = 15
DATASETS = ["20NG", "AGNews", "IMDB", "YahooAnswer"]
K_VALUES = [50, 100]


lr = 2e-4        
epochs = 200     
batch_size = 100 
hidden_size = 1024 

os.makedirs(WLDA_ROOT, exist_ok=True)
print(f"Dossier WLDA dedie: {WLDA_ROOT}")

Dossier WLDA dedie: C:\Users\Home\Documents\M2\Projet_PPD\output\WLDA


### Fonctions utilitaires

In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # comportement déterministe
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_dataset(name: str):
    path = os.path.join(DATA_ROOT, name)
    train_bow = sp.load_npz(os.path.join(path, "train_bow.npz")).toarray()
    test_bow  = sp.load_npz(os.path.join(path, "test_bow.npz")).toarray()

    with open(os.path.join(path, "vocab.txt"), encoding="utf-8") as f:
        vocab = [w.strip() for w in f]

    labels = np.loadtxt(os.path.join(path, "train_labels.txt"), dtype=int)

    return train_bow, test_bow, vocab, labels

SEED = 42
set_seed(SEED)

print("Fonctions prêtes ")


Fonctions prêtes 


### Implémentation du modèle WLDA

In [4]:
class WLDA(nn.Module):
    def __init__(self, vocab_size, num_topics, hidden_size=512):  # ↑512
        super().__init__()
        self.num_topics = num_topics
        self.vocab_size = vocab_size
        

        self.encoder = nn.Sequential(
            nn.Linear(vocab_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_topics),
            nn.Softplus() 
        )
        
        self.beta = nn.Parameter(torch.randn(num_topics, vocab_size) * 0.001)  # ↓0.01→0.001
        
    def encode(self, x):
        alpha = self.encoder(x) + 1e-6
        return alpha
    
    def reparameterize(self, alpha):
        dist = Dirichlet(alpha)
        theta = dist.rsample()
        return theta
    
    def decode(self, theta):
        logits = torch.matmul(theta, self.beta)
        return F.log_softmax(logits, dim=-1)
    
    def forward(self, x):
        alpha = self.encode(x)
        theta = self.reparameterize(alpha)
        recon_loss = -(x * self.decode(theta)).sum(dim=-1).mean()
        
        kl_loss = torch.distributions.kl_divergence(
            Dirichlet(alpha), 
            Dirichlet(torch.ones_like(alpha))
        ).mean()
        
        return recon_loss + 0.05 * kl_loss  # ↓0.1→0.05

    def get_beta(self):
        return F.softmax(self.beta, dim=-1)
    
    def get_theta(self, bow):
        alpha = self.encode(bow)
        dist = Dirichlet(alpha)
        return dist.rsample()

print("Modele WLDA defini")


Modele WLDA defini


### Fonctions métriques

In [5]:
def topic_diversity(topics):
    word_freq = collections.Counter()
    for topic in topics:
        word_freq.update(topic)
    exclusifs = sum(1 for count in word_freq.values() if count == 1)
    return exclusifs / len(word_freq) if word_freq else 0.0

def purity_score(y_true, y_pred):
    total = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        labels, counts = np.unique(y_true[idx], return_counts=True)
        total += counts.max()
    return total / len(y_true)

def compute_nmi(theta, labels):
    pred = theta.argmax(axis=1)
    return sklearn.metrics.normalized_mutual_info_score(labels, pred)


In [20]:
def evaluate_wlda_full(dataset, K, seed=42, topn=15, run_palmetto=True):
    dataset_dir = f"{WLDA_ROOT}/{dataset}"
    mat_path = f"{dataset_dir}/{dataset}_WLDA_K{K}_seed{seed}.mat"
    
    if not os.path.isfile(mat_path):
        print(f" Fichier .mat absent : {mat_path}")
        return None
    
    # 1. Chargement unique des données
    loaded = scipy.io.loadmat(mat_path)
    beta = loaded["beta"]
    train_theta = loaded["train_theta"]
    _, _, vocab, labels = load_dataset(dataset)
    
    # 2. Calcul des métriques internes (TD, Purity, NMI)
    topics = [[vocab[i] for i in np.argsort(beta[k])[::-1][:topn]] for k in range(beta.shape[0])]
    td = topic_diversity(topics)
    
    y_pred = train_theta.argmax(axis=1)
    pur = purity_score(labels, y_pred)
    nmi = compute_nmi(train_theta, labels)
    
    print(f"{dataset} (K={K}) | TD: {td:.4f} | Pur: {pur:.4f} | NMI: {nmi:.4f}")
    
    # 4. Exportation unique
    out_path = f"{dataset_dir}/full_metrics_{dataset}_K{K}.txt"
    with open(out_path, "w") as f:
        f.write(f"Dataset: {dataset}\nK: {K}\nTD: {td}\nPurity: {pur}\nNMI: {nmi}")
        
    return td, pur, nmi

### Boucle d'entraînement du modèle

In [7]:
def train_one_wlda(dataset, K, seed=42):
    set_seed(seed)
    
    dataset_dir = f"{WLDA_ROOT}/{dataset}"
    os.makedirs(dataset_dir, exist_ok=True)
    
    model_path = f"{dataset_dir}/WLDA_{dataset}_K{K}_seed{seed}.pt"
    mat_path = f"{dataset_dir}/{dataset}_WLDA_K{K}_seed{seed}.mat"
    
    if os.path.exists(mat_path):
        print(f"Deja existant: {mat_path}")
        return mat_path
    
    print(f"WLDA | {dataset} | K={K}")
    
    # Données
    train_bow, test_bow, vocab, _ = load_dataset(dataset)
    V = train_bow.shape[1]
    
    # Modèle 
    model = WLDA(V, K, hidden_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Dataset 
    X_train_t = torch.tensor(train_bow, dtype=torch.float32)
    loader = DataLoader(
        TensorDataset(X_train_t),
        batch_size=batch_size,
        shuffle=True
    )
    
    # Entraînement
    epoch_bar = tqdm(range(epochs), desc=f"WLDA_{dataset}_K{K}")
    for epoch in epoch_bar:
        model.train()
        total_loss = 0.0
        
        for (batch,) in loader:
            optimizer.zero_grad()
            loss = model(batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        
        epoch_bar.set_postfix(loss=f"{total_loss / len(loader):.2f}")
    
    # Extraire beta / theta (CPU)
    model.eval()
    with torch.no_grad():
        beta_np = model.get_beta().numpy()
        
        X_train = torch.tensor(train_bow, dtype=torch.float32)
        train_theta = model.get_theta(X_train).numpy()
        
        X_test = torch.tensor(test_bow, dtype=torch.float32)
        test_theta = model.get_theta(X_test).numpy()
    
    scipy.io.savemat(mat_path, {
        "beta": beta_np.astype(np.float32),
        "train_theta": train_theta.astype(np.float32),
        "test_theta": test_theta.astype(np.float32)
    })
    
    torch.save(model.state_dict(), model_path)
    print(f"Sauvegarde: {mat_path}")
    return mat_path


### Exécution globale de WLDA

In [8]:
print("=" * 60)
print("ENTRAINEMENT WLDA - 4 BASES")
print("=" * 60)

total_processed = 0
total_skipped = 0
total_models = len(DATASETS) * len(K_VALUES)

with tqdm(total=total_models, desc="WLDA Global") as pbar:
    for dataset in DATASETS:
        print(f"\n{'='*50}")
        print(f"BASE: {dataset}")
        
        with tqdm(total=len(K_VALUES), desc=f"{dataset}", leave=False) as pbar_k:
            for K in K_VALUES:
                dataset_dir = f"{WLDA_ROOT}/{dataset}"
                mat_path = f"{dataset_dir}/{dataset}_WLDA_K{K}_seed{SEED}.mat"
                
                print(f"\n--- {dataset} K={K} ---")
                
                if os.path.exists(mat_path):
                    print(f"SKIP (deja calcule)")
                    total_skipped += 1
                else:
                    mat_path = train_one_wlda(dataset, K)
                    total_processed += 1
                
                pbar_k.update(1)
                pbar.update(1)
                pbar.set_postfix({'processed': total_processed, 'skipped': total_skipped})

print(f"\nTraitees: {total_processed}, Sautees: {total_skipped}")


ENTRAINEMENT WLDA - 4 BASES


WLDA Global:   0%|          | 0/8 [00:00<?, ?it/s]


BASE: 20NG


20NG:   0%|          | 0/2 [00:00<?, ?it/s]


--- 20NG K=50 ---
SKIP (deja calcule)

--- 20NG K=100 ---
SKIP (deja calcule)

BASE: AGNews


...


--- AGNews K=50 ---
SKIP (deja calcule)

--- AGNews K=100 ---
SKIP (deja calcule)

BASE: IMDB


IMDB:   0%|          | 0/2 [00:00<?, ?it/s]


--- IMDB K=50 ---
SKIP (deja calcule)

--- IMDB K=100 ---
SKIP (deja calcule)

BASE: YahooAnswer


YahooAnswer:   0%|          | 0/2 [00:00<?, ?it/s]


--- YahooAnswer K=50 ---
SKIP (deja calcule)

--- YahooAnswer K=100 ---
SKIP (deja calcule)

Traitees: 0, Sautees: 8


### Calcul des métriques

In [21]:
WLDA_ROOT = r"C:\Users\Home\Documents\M2\Projet_PPD\output\WLDA"

for dataset in DATASETS:
    print(f"\n{'='*50}\nBASE: {dataset}\n{'='*50}")
    
    for K in K_VALUES:
        filename = f"metrics_{dataset}_WLDA_K{K}_seed42.csv"
        filepath = os.path.join(WLDA_ROOT, dataset, filename)
        

        if os.path.exists(filepath):
            print(f" [SKIP] {dataset} K={K} : Déjà présent dans output.")
            continue
            

        print(f" [CALCUL] Lancement WLDA pour {dataset} K={K}...")

        evaluate_wlda_full(dataset, K)


BASE: 20NG
 [CALCUL] Lancement WLDA pour 20NG K=50...
20NG (K=50) | TD: 0.4865 | Pur: 0.1680 | NMI: 0.2589
 [CALCUL] Lancement WLDA pour 20NG K=100...
20NG (K=100) | TD: 0.4103 | Pur: 0.1883 | NMI: 0.2545

BASE: AGNews
 [CALCUL] Lancement WLDA pour AGNews K=50...
AGNews (K=50) | TD: 0.4595 | Pur: 0.6579 | NMI: 0.2881
 [CALCUL] Lancement WLDA pour AGNews K=100...
AGNews (K=100) | TD: 0.0500 | Pur: 0.3165 | NMI: 0.0092

BASE: IMDB
 [CALCUL] Lancement WLDA pour IMDB K=50...
IMDB (K=50) | TD: 0.3065 | Pur: 0.6839 | NMI: 0.0758
 [CALCUL] Lancement WLDA pour IMDB K=100...
IMDB (K=100) | TD: 0.3857 | Pur: 0.6828 | NMI: 0.0654

BASE: YahooAnswer
 [CALCUL] Lancement WLDA pour YahooAnswer K=50...
YahooAnswer (K=50) | TD: 0.3939 | Pur: 0.2857 | NMI: 0.1376
 [CALCUL] Lancement WLDA pour YahooAnswer K=100...
YahooAnswer (K=100) | TD: 0.3333 | Pur: 0.3480 | NMI: 0.1605


### Calcul de la Cohérence Sémantique

In [13]:
def compute_gensim_cv_wlda_dataset(dataset, K, topn=15):
    print(f"\n=== WLDA {dataset} K={K} — C_V (gensim) ===")

    # 1) Charger BoW + vocab
    train_bow, test_bow, vocab, labels = load_dataset(dataset)  

    # 2) Construire textes tokenisés à partir de train_bow
    tokenized_texts = []
    for doc in train_bow:
        words = [vocab[i] for i, freq in enumerate(doc) if freq > 0]
        if len(words) >= 2:
            tokenized_texts.append(words)

    print(f"Corpus pour Gensim: {len(tokenized_texts)} docs, {len(vocab)} mots")

    # 3) Charger beta depuis le .mat WLDA
    dataset_dir = f"{WLDA_ROOT}\{dataset}"
    mat_path = f"{dataset_dir}\{dataset}_WLDA_K{K}_seed{SEED}.mat"
    if not os.path.isfile(mat_path):
        print(f" Pas de .mat pour {dataset} K={K}: {mat_path}")
        return None

    loaded = scipy.io.loadmat(mat_path)
    beta = loaded["beta"]  # shape (K, V)
    print(f"beta: {beta.shape}")

    # 4) Construire les topics (listes de mots) pour Gensim
    wlda_topics = []
    for k in range(beta.shape[0]):
        top_idx = beta[k].argsort()[-topn:][::-1]  # top-n indices
        topic_words = [vocab[i] for i in top_idx if i < len(vocab)]
        if len(topic_words) >= 2:
            wlda_topics.append(topic_words)

    print(f"{len(wlda_topics)} topics construits (top{topn})")

    # 5) Dictionnaire + modèle de cohérence C_V
    dictionary = Dictionary(tokenized_texts)
    cm = CoherenceModel(
        topics=wlda_topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence="c_v",   
    )

    cv_score = cm.get_coherence()
    print(f"C_V (Gensim) WLDA {dataset} K={K} = {cv_score:.4f}")
    return cv_score

In [19]:
OUTPUT_BASE = r"C:\Users\Home\Documents\M2\Projet_PPD\output\WLDA"

wlda_cv_results = []

for dataset in DATASETS:
    dataset_dir = os.path.join(OUTPUT_BASE, dataset)
    
    if not os.path.exists(dataset_dir):
        os.makedirs(dataset_dir, exist_ok=True)
    
    for K in K_VALUES:
        filename = f"palmetto_CV_{dataset}_WLDA_K{K}_seed42.csv"
        filepath = os.path.join(dataset_dir, filename)
        
        if os.path.exists(filepath):
            try:
                df_tmp = pd.read_csv(filepath)
                cv_val = df_tmp['C_V'].iloc[0]
                wlda_cv_results.append({"Dataset": dataset, "K": K, "C_V": cv_val})
                print(f" [SKIP] {dataset} K={K} : Trouvé dans {dataset}/")
                continue
            except Exception:
                print(f" [RE-CALCUL] Fichier corrompu ou illisible : {filename}")

        print(f" Calcul C_V pour {dataset} K={K}...")
        cv = compute_gensim_cv_wlda_dataset(dataset, K, topn=TOPN)
        
        if cv is not None:
            wlda_cv_results.append({"Dataset": dataset, "K": K, "C_V": cv})
            res_df = pd.DataFrame([{"Dataset": dataset, "K": K, "C_V": cv}])
            res_df.to_csv(filepath, index=False)
            print(f" -> Sauvegardé: {dataset}/{filename}")

if wlda_cv_results:
    final_df = pd.DataFrame(wlda_cv_results)
    print("\n" + "="*50)
    print(f"{'TABLEAU RÉCAPITULATIF WLDA - C_V':^50}")
    print("="*50)
    summary = final_df.pivot(index="Dataset", columns="K", values="C_V")
    print(summary.round(4))
    print("="*50)

 [SKIP] 20NG K=50 : Trouvé dans 20NG/
 [SKIP] 20NG K=100 : Trouvé dans 20NG/
 [SKIP] AGNews K=50 : Trouvé dans AGNews/
 [SKIP] AGNews K=100 : Trouvé dans AGNews/
 [SKIP] IMDB K=50 : Trouvé dans IMDB/
 [SKIP] IMDB K=100 : Trouvé dans IMDB/
 [SKIP] YahooAnswer K=50 : Trouvé dans YahooAnswer/
 [SKIP] YahooAnswer K=100 : Trouvé dans YahooAnswer/

         TABLEAU RÉCAPITULATIF WLDA - C_V         
K               50      100
Dataset                    
20NG         0.5225  0.5298
AGNews       0.2606  0.2611
IMDB         0.2635  0.2830
YahooAnswer  0.5856  0.6469


### Analyse comparative : Résultats originaux vs Reproduction

In [22]:
# Données du Papier 
paper_wlda = {
    "20NG": {
        "K50":  {"C_V": 0.523, "TD": 0.486, "Purity": 0.168, "NMI": 0.259},
        "K100": {"C_V": 0.530, "TD": 0.410, "Purity": 0.188, "NMI": 0.254}
    },
    "IMDB": {
        "K50":  {"C_V": 0.264, "TD": 0.306, "Purity": 0.684, "NMI": 0.076},
        "K100": {"C_V": 0.283, "TD": 0.386, "Purity": 0.683, "NMI": 0.065}
    },
    "YahooAnswer": {
        "K50":  {"C_V": 0.586, "TD": 0.394, "Purity": 0.286, "NMI": 0.138},
        "K100": {"C_V": 0.647, "TD": 0.333, "Purity": 0.348, "NMI": 0.161}
    },
    "AGNews": {
        "K50":  {"C_V": 0.261, "TD": 0.459, "Purity": 0.658, "NMI": 0.288},
        "K100": {"C_V": 0.261, "TD": 0.050, "Purity": 0.317, "NMI": 0.009}
    }
}

# Résultats de Reproduction 
reproduction_wlda = {
    "20NG": {
        "K50":  {"C_V": 0.378, "TD": 0.375, "Purity": 0.233, "NMI": 0.157},
        "K100": {"C_V": 0.369, "TD": 0.273, "Purity": 0.292, "NMI": 0.207}
    },
    "IMDB": {
        "K50":  {"C_V": 0.311, "TD": 0.053, "Purity": 0.589, "NMI": 0.011},
        "K100": {"C_V": 0.320, "TD": 0.069, "Purity": 0.602, "NMI": 0.013}
    },
    "YahooAnswer": {
        "K50":  {"C_V": 0.333, "TD": 0.322, "Purity": 0.255, "NMI": 0.084},
        "K100": {"C_V": 0.338, "TD": 0.292, "Purity": 0.303, "NMI": 0.127}
    },
    "AGNews": {
        "K50":  {"C_V": 0.384, "TD": 0.410, "Purity": 0.580, "NMI": 0.151},
        "K100": {"C_V": 0.378, "TD": 0.323, "Purity": 0.653, "NMI": 0.188}
    }
}

data = []
for dataset in ["20NG", "IMDB", "YahooAnswer", "AGNews"]:
    for k_str in ["K50", "K100"]:
        p, r = paper_wlda[dataset][k_str], reproduction_wlda[dataset][k_str]
        row = {"Dataset": dataset, "K": int(k_str[1:])}
        for m in ["C_V", "TD", "Purity", "NMI"]:
            row[f"{m}_Repro"]  = r[m]  # Reproduction à gauche
            row[f"{m}_Papier"] = p[m]  # Papier à droite
            row[f"{m}_Ecart"]  = round(r[m] - p[m], 3)
        data.append(row)

df = pd.DataFrame(data).set_index(["Dataset", "K"])



df.style\
    .format("{:.3f}")\
    .set_caption("WLDA — Reproduction vs Papier Original (Écarts)")